# SkinToneNet — HAM10000 Ablation (Kaggle T4 GPU)
Trains `baseline`, `aug_only`, `tone_only` variants and saves checkpoints to `/kaggle/working/`.

**Required datasets (add via right-hand panel → Add Data):**
- `kmader/skin-cancer-mnist-ham10000`  (HAM10000 images + metadata)
- `aaronajit/skintone-script`  (contains `skintone.py`)

**Runtime:** ~6 h on T4 for 3 variants × 40 epochs.

In [ ]:
# Cell 1 — Install missing packages
# Kaggle ships numpy 2.x + pandas compiled against numpy 2.x — DO NOT downgrade numpy.
# Only timm is missing from Kaggle's pre-installed environment.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
print('timm installed.')

In [ ]:
# Cell 2 — Copy skintone.py from dataset input
import shutil, os, glob

# Try the named dataset first, then fall back to any skintone.py in /kaggle/input
candidates = glob.glob('/kaggle/input/**/skintone.py', recursive=True)
if candidates:
    shutil.copy(candidates[0], '/kaggle/working/skintone.py')
    print(f'Copied from: {candidates[0]}')
else:
    raise FileNotFoundError(
        'skintone.py not found in /kaggle/input. '
        'Add the dataset containing skintone.py via Add Data.'
    )
print('skintone.py ready:', os.path.exists('/kaggle/working/skintone.py'))

In [ ]:
# Cell 3 — Verify environment
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torchvision
import timm
import sklearn
import scipy
from pathlib import Path
import glob as _glob

print(f'numpy:       {np.__version__}')
print(f'pandas:      {pd.__version__}')
print(f'torch:       {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'timm:        {timm.__version__}')
print(f'GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — change to GPU accelerator"}')

# Show what datasets are actually mounted so we can see the real slug
print('\n/kaggle/input contents:')
for p in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {p}')

# Auto-discover HAM10000 by finding all .jpg files under /kaggle/input
jpg_files = _glob.glob('/kaggle/input/**/*.jpg', recursive=True)
n_imgs = len(jpg_files)
print(f'\nTotal .jpg files found: {n_imgs}')
assert n_imgs > 9000, f'Expected >9000 images, got {n_imgs}. Is the HAM10000 dataset attached?'

# Walk up from an image file until we find the directory containing the metadata CSV
ham_dir = Path(jpg_files[0]).parent
while not list(ham_dir.glob('*metadata*.csv')) and ham_dir != Path('/kaggle/input'):
    ham_dir = ham_dir.parent
print(f'ham_dir resolved to: {ham_dir}')

# Make ham_dir available to later cells
import builtins
builtins.HAM_DIR = str(ham_dir)
print('All checks passed.')

In [ ]:
# Cell 4 — Train baseline_balanced (~2 h on T4)
# Tests whether the dark-skin AUC gap is a data-counting artefact or a
# feature-learning problem. Identical to baseline except dark/medium tone
# groups are oversampled to match light-skin count in training.
import subprocess, sys, os, builtins

os.chdir('/kaggle/working')
ham_dir = builtins.HAM_DIR  # set by Cell 3

proc = subprocess.Popen(
    [
        sys.executable, 'skintone.py',
        '--mode',       'balanced',
        '--ham_dir',    ham_dir,
        '--output_dir', '/kaggle/working',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')
if proc.returncode != 0:
    raise RuntimeError('Training failed — see output above.')

In [ ]:
# Cell 5 — List output files
import os
from pathlib import Path

print('Files in /kaggle/working:')
for f in sorted(os.listdir('/kaggle/working')):
    size = os.path.getsize(f'/kaggle/working/{f}')
    print(f'  {f:<45}  {size/1024/1024:.1f} MB' if size > 1024*1024 else f'  {f:<45}  {size/1024:.0f} KB')

ckpt = Path('/kaggle/working/baseline_balanced_best.pt')
status = f'OK ({ckpt.stat().st_size/1024/1024:.0f} MB)' if ckpt.exists() else 'MISSING'
print(f'\nbaseline_balanced_best.pt: {status}')